# ClinIQ — Gemma 4 E2B fine-tuned with Unsloth for offline eICR extraction

**Experiment branch v63b (fix LoRA k/v coverage regression on global-attention layers 15-34). v62 remains the shipped submission; v63 produced a partial-coverage adapter (410 tensors vs v62's 490) that survived full-precision bench (F1 0.9989) but collapsed to F1 0.5475 once Q3_K_M-quantized for Mac. v63b forces full k/v coverage on all 35 decoder layers (local 0-14 + global 15-34).**

This notebook trains a LoRA adapter on `unsloth/gemma-4-E2B-it` end-to-end on Kaggle's free T4 x2 in about 1 hour 4 minutes, runs an inline 200-case held-out bench, and exports the 124 MB adapter to the kernel Output tab for offline edge deployment.

## v63 -> v63b fix summary

- Diagnosis: v63 saved 410 LoRA tensors; v62 saved 490. The 80-tensor delta == 20 global-attention layers (15-34) x 2 modules (k_proj, v_proj) x 2 matrices (A, B). adapter_config.json claimed full coverage (`layers_to_transform=null`, all 7 target_modules), but a newer peft/unsloth release silently dropped k/v on global-attention layers. The "unsloth_fixed": true line in the auto_mapping was the smoking gun.
- Fix (this notebook, v63b):
  1. `get_peft_model(..., layers_to_transform=list(range(35)))` set explicitly.
  2. Post-LoRA-setup assertion counts wrapped k_proj/v_proj LoRA submodules; must be 35 each or `sys.exit(1)`.
  3. Post-save assertion counts safetensors tensors; must be >= 490 or fail.
  4. Optional fallback install path pinned to `peft<0.18` and `unsloth<2026.5.0` (set env `V63B_PIN_OLD=1` to enable).
- Acceptance: Mac Q3_K_M F1 >= 0.85 after llama.cpp conversion (vs v63's 0.5475).

## Unsloth Track eligibility — what Unsloth APIs are used and why

This notebook is built directly on the Unsloth library. The pipeline below uses three Unsloth-specific APIs, each of which contributes a measurable benefit (not a stylistic choice or a thin wrapper):

| Unsloth API | Where in this notebook | What it does for us |
|---|---|---|
| `unsloth.FastLanguageModel.from_pretrained(model_name="unsloth/gemma-4-E2B-it", load_in_4bit=True)` | Section 2 (model load) | 4-bit base fits comfortably on a Kaggle T4 (16 GB) with headroom. Vanilla HF + bitsandbytes works but needs more careful manual config. |
| `unsloth.FastLanguageModel.get_peft_model(..., use_gradient_checkpointing="unsloth")` | Section 3 (LoRA setup) | Unsloth's checkpointing recipe drops peak VRAM ~30% vs vanilla `gradient_checkpointing=True`, which is what lets us train at `max_seq_length=1024` with `packing=True` on a T4. |
| `unsloth.chat_templates.get_chat_template(tokenizer, "gemma-4")` | Section 5 (formatting) | Gemma 4's native chat template uses `<|turn>system\n...<turn|>\n` markers that the HF tokenizer's default doesn't emit. Unsloth ships the verified template inline; vanilla HF tokenizer would have produced `<start_of_turn>` / `<end_of_turn>` and the LoRA would have learned the wrong delimiters, breaking inference. |

There is also one Unsloth helper used at inference time:

- `FastLanguageModel.for_inference(model)` — switches the LoRA-augmented model into the optimized inference path used by the inline single-example test (Section 7).

**Why the choice of `unsloth/gemma-4-E2B-it` specifically:** the Unsloth-shipped 4-bit weight is a clean drop-in for the official Google checkpoint and includes the chat-template fixes needed for `<|turn>` parsing on Gemma 4. The base model is unmodified Gemma 4 E2B-it; the LoRA adapter is the only artifact this notebook produces.


## 1. Install dependencies

Unsloth pulls compatible versions of `trl`, `peft`, `accelerate`, and `bitsandbytes`. We install with `--no-deps` for the latter set so we don't drag in newer transformers releases that conflict with Unsloth's pin.


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # force single GPU

# v63b: two install variants gated by env var V63B_PIN_OLD.
#   default (V63B_PIN_OLD unset/"0"): latest unsloth + transformers>=5.5,<=5.5.0 + --no-deps for trl/peft/accelerate/bitsandbytes
#                                    + explicit layers_to_transform fix in Section 3 forces full k/v coverage.
#   V63B_PIN_OLD=1: pin older peft<0.18 and unsloth<2026.5.0 as a fallback in case the layers_to_transform
#                  fix alone doesn't restore coverage (suspected regression in newer peft/unsloth).
PIN_OLD = os.environ.get("V63B_PIN_OLD", "0") == "1"
print(f"V63B install variant: {'PIN_OLD' if PIN_OLD else 'LATEST + layers_to_transform fix'}")

if PIN_OLD:
    # Fallback path: pin older versions that did not have the k/v coverage regression.
    !pip install -q "unsloth<2026.5.0"
    !pip install -q --upgrade "transformers>=5.5,<=5.5.0"
    !pip install -q --no-deps trl "peft<0.18" accelerate bitsandbytes
else:
    # Default path: latest unsloth + the explicit layers_to_transform fix in Section 3.
    # Full install (with deps) so torch lands on a version aligned with unsloth.
    # Then explicitly upgrade transformers — Kaggle's preinstalled 5.0.0 does not know
    # Gemma 4 (`model_type=gemma4` -> KeyError); >=5.5 is required.
    !pip install -q unsloth
    !pip install -q --upgrade "transformers>=5.5,<=5.5.0"
    !pip install -q --no-deps trl peft accelerate bitsandbytes

import torch, sys
print("python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.version.cuda,
      "device", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no-gpu",
      "cc", torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "n/a")
import importlib.metadata as md
for pkg in ("unsloth","unsloth_zoo","transformers","trl","peft","bitsandbytes","accelerate"):
    try: print(pkg, md.version(pkg))
    except Exception as e: print(pkg, "NOT INSTALLED", e)
assert torch.cuda.is_available(), "no CUDA device — kernel needs a GPU runtime"
cap = torch.cuda.get_device_capability(0)
assert cap >= (7, 5), f"GPU compute capability {cap} too low; expected T4 sm_75 or newer"


## Background — eICR / EZeCR / public-health context

An eICR (electronic Initial Case Report) is the HL7 CDA document a US clinician (or EHR) sends to a state public-health authority when a patient presents with a CDC RCTC reportable condition (COVID-19, influenza, measles, pertussis, TB, HIV, syphilis, hepatitis A/B/C, streptococcus, etc.). The current production path runs through CDC's **EZeCR** service, which calls AWS Comprehend Medical for clinical NER and IMO for ontology mapping (SNOMED CT, ICD-10-CM, LOINC, RxNorm). Both are cloud, both require PHI to leave the clinic, both cost real money per call.

The Gemma 4 Good Hackathon's "low-resource clinic" framing maps cleanly onto this gap: an LMIC clinic, a field hospital, or a disaster-response site rarely has reliable internet for cloud NLP, and the regulatory case for keeping PHI local is strong. A single-shot edge-deployable extractor that fits in ~5 GB (base model + 124 MB LoRA at Q3_K_M) and runs at ~4s p50 on a Mac M-series — or seconds on Jetson Orin — is the missing piece.

This notebook produces that extractor's training run. The repo (`patrickdeutsch/gemma4`) ships the surrounding plumbing: GBNF grammar, GGUF conversion, an iOS app, a Spaces demo, and a verified-RAG agent path that hits F1 = 1.000 on a 64-case combined eICR set for harder out-of-distribution narrative inputs.


## 2. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024  # Compact output ~150 tokens + prompt ~150 = ~300 total
dtype = None  # auto-detect (bfloat16 on A100)
load_in_4bit = True  # QLoRA for memory efficiency

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 3. Configure LoRA Adapters

In [ ]:
# v63b: force full k/v coverage on ALL 35 decoder layers (local 0-14 + global 15-34).
# Gemma 4 E2B has 35 decoder layers; v63 (latest peft/unsloth) silently dropped k_proj/v_proj
# LoRA on the global-attention layers (15-34), saving only 410 tensors vs v62's 490.
# We pass layers_to_transform=list(range(35)) explicitly so peft cannot infer a partial range.
import sys

NUM_DECODER_LAYERS = 35  # Gemma 4 E2B

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    layers_to_transform=list(range(NUM_DECODER_LAYERS)),  # v63b fix: explicit full coverage
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
)

# ----- v63b coverage assertion: fail fast if k/v are missing on any layer ----
# We count distinct decoder-layer indices that have a LoRA-wrapped k_proj and v_proj submodule.
# Gemma 4 hybrid-attention design uses local attention 0-14 + global attention 15-34; the v63 bug
# dropped LoRA on the global half. Anything less than 35 here = the bug is still present and we
# should not waste a training run.
import re
from collections import defaultdict

per_module_layers = defaultdict(set)
for name, module in model.named_modules():
    # peft wraps target modules as LoraLayer instances containing lora_A / lora_B ModuleDicts.
    if not (hasattr(module, "lora_A") and hasattr(module, "lora_B")):
        continue
    # name looks like:
    #   base_model.model.model.layers.17.self_attn.k_proj
    m = re.search(r"\.layers\.(\d+)\.[^.]+\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$", name)
    if not m:
        continue
    layer_idx = int(m.group(1))
    mod_name = m.group(2)
    # Confirm at least one active adapter has both A and B matrices materialized.
    a_keys = list(getattr(module, "lora_A", {}).keys())
    b_keys = list(getattr(module, "lora_B", {}).keys())
    if a_keys and b_keys:
        per_module_layers[mod_name].add(layer_idx)

print("\n=== v63b LoRA coverage report ===")
for mod_name in ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"):
    layers = sorted(per_module_layers.get(mod_name, set()))
    missing = sorted(set(range(NUM_DECODER_LAYERS)) - set(layers))
    print(f"  {mod_name:10s} layers wrapped: {len(layers)}/{NUM_DECODER_LAYERS}  missing={missing if missing else 'none'}")

k_count = len(per_module_layers.get("k_proj", set()))
v_count = len(per_module_layers.get("v_proj", set()))
print(f"\nk_proj wrapped on {k_count}/{NUM_DECODER_LAYERS} layers")
print(f"v_proj wrapped on {v_count}/{NUM_DECODER_LAYERS} layers")

if k_count < NUM_DECODER_LAYERS or v_count < NUM_DECODER_LAYERS:
    print(f"\nFATAL: v63b coverage check FAILED — k_proj={k_count}, v_proj={v_count}, expected {NUM_DECODER_LAYERS} each.", flush=True)
    print("This is the v63 bug. Re-run with env V63B_PIN_OLD=1 to try the pinned-older fallback path.", flush=True)
    sys.exit(1)

print(f"\nv63b coverage OK — full k/v coverage on all {NUM_DECODER_LAYERS} decoder layers.")

## 4. Load Training Data

In [ ]:
from datasets import load_dataset

import os

COMPACT = True  # Set to True for compact output (fewer tokens, faster inference)
DATASET_PATH = "/kaggle/input/eicr-fhir-training-data"

# Debug: list all available Kaggle inputs
input_dir = "/kaggle/input"
if os.path.exists(input_dir):
    print(f"Kaggle inputs: {os.listdir(input_dir)}")
    for d in os.listdir(input_dir):
        full = os.path.join(input_dir, d)
        if os.path.isdir(full):
            print(f"  {d}/: {os.listdir(full)}")

# Fallback: try alternative Kaggle mount paths
if not os.path.exists(DATASET_PATH):
    for alt in ["/kaggle/input/datasets/patrickdeutsch/eicr-fhir-training-data",
                "/kaggle/input/datasets"]:
        if os.path.exists(alt):
            DATASET_PATH = alt
            print(f"Using alternative path: {DATASET_PATH}")
            break

suffix = "-compact" if COMPACT else ""
train_file = f"{DATASET_PATH}/train{suffix}.jsonl"
val_file = f"{DATASET_PATH}/val{suffix}.jsonl"
if not os.path.exists(train_file):
    print(f"WARNING: {train_file} not found, trying verbose")
    COMPACT = False
    suffix = ""
    train_file = f"{DATASET_PATH}/train.jsonl"
    val_file = f"{DATASET_PATH}/val.jsonl"

dataset = load_dataset("json", data_files={
    "train": train_file,
    "validation": val_file,
})

print(f"Dataset: {DATASET_PATH}")
print(f"Train: {len(dataset['train'])}  Val: {len(dataset['validation'])}")
# Verify we loaded the right data
sample_prompt = dataset['train'][0]['conversations'][0]['content']
print(f"System prompt: {sample_prompt[:80]}...")
assert "No summary" in sample_prompt or not COMPACT, "COMPACT=True but loaded verbose data!"

## 5. Format Dataset for Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-4",
)


def format_prompts(examples):
    texts = []
    for convos in examples["conversations"]:
        text = tokenizer.apply_chat_template(
            convos,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}


dataset = dataset.map(format_prompts, batched=True)

## 6. Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=5,
        learning_rate=1e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

In [ ]:
import torch
props = torch.cuda.get_device_properties(0)
total_mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
print(f"GPU = {torch.cuda.get_device_name(0)}. Max memory = {total_mem / 1024**3:.3f} GB.")
print(f"{torch.cuda.memory_reserved() / 1024**3:.3f} GB of memory reserved.")

trainer_stats = trainer.train()
runtime = trainer_stats.metrics['train_runtime']
print(f"{runtime:.0f}s training time")
peak = torch.cuda.max_memory_reserved() / 1024**3
used = (torch.cuda.max_memory_reserved() - torch.cuda.memory_reserved()) / 1024**3
print(f"Peak memory = {peak:.3f} GB (training used {used:.3f} GB)")
print(f"Loss: {trainer_stats.training_loss:.4f}")
if runtime < 600:
    print("Quick proof complete — increase max_steps for full training")

## 7. Test Inference

In [ ]:
# Quick inference test (wrapped in try/except so save always runs)
try:
    FastLanguageModel.for_inference(model)
    test_input = "Patient: Maria Garcia\nGender: F\nDOB: 1985-06-14\nLocation: Denver, CO 80202\nDx: COVID-19 (SNOMED 840539006)\nLab: SARS-CoV-2 RNA (LOINC 94500-6) - Detected\nMeds: nirmatrelvir (RxNorm 2599543)"
    sys_prompt = "Extract clinical entities from this eICR. Output compact JSON with: patient, encounter, conditions (SNOMED), labs (LOINC), meds (RxNorm), vitals. No summary. Valid JSON only." if COMPACT else "Extract clinical entities from this eICR summary. Output JSON with: patient demographics, conditions (SNOMED/ICD-10), labs (LOINC), medications (RxNorm), vitals, and a case summary. Include confidence scores. Output valid JSON only."
    inputs = tokenizer.apply_chat_template([{"role": "system", "content": sys_prompt}, {"role": "user", "content": test_input}], tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.1, top_p=0.9)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print("=== Model output ===")
    print(response[:500])
except Exception as e:
    print(f"Inference test failed: {e}")

## 8. Inline bench on val-compact (200 held-out cases)

We score the LoRA-augmented model on the held-out 200-case `val-compact.jsonl` split — the same split the offline llama.cpp bench at `apps/mobile/convert/bench_v62_singleshot.py` uses. Code-level micro-F1 is computed on the union of `(coding_system, code)` tuples across `conditions / labs / meds`, with `vitals` rolled in as presence-only pseudo-codes so missing vitals are penalised. JSON validity is reported alongside.

This cell can be slow (200 generations on Kaggle T4 takes ~15-25 min depending on average output length). To preview without committing the full sweep, set `BENCH_LIMIT = 25` below.


In [ ]:
# Inline bench: score the LoRA on val-compact end-to-end inside this kernel.
# Mirrors the scoring logic in apps/mobile/convert/bench_v62_singleshot.py
# from the project repo (https://github.com/patrickdeutsch/gemma4).
import json
import re
import statistics
import time
from pathlib import Path

import torch

# ----- config ---------------------------------------------------------------
BENCH_LIMIT = None   # set to e.g. 25 for a quick preview, None = all 200
BENCH_MAX_NEW_TOKENS = 1024
SCHEMA_KEYS = {"patient", "encounter", "conditions", "labs", "meds", "vitals"}
CODE_SYSTEMS_LIST = [
    ("snomed", "snomed"),
    ("icd10", "icd10"),
    ("loinc", "loinc"),
    ("rxnorm", "rxnorm"),
]

# Locate the val file. We use the same Kaggle dataset attachment as training.
val_path = Path(f"{DATASET_PATH}/val-compact.jsonl")
if not val_path.exists():
    val_path = Path(f"{DATASET_PATH}/val.jsonl")
print(f"Bench val file: {val_path}")


# ----- scoring helpers ------------------------------------------------------
def try_parse_json(text):
    if not text:
        return None
    s = text.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json)?\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    try:
        return json.loads(s)
    except Exception:
        pass
    start = s.find("{")
    if start < 0:
        return None
    depth, in_str, esc = 0, False, False
    for i in range(start, len(s)):
        ch = s[i]
        if esc:
            esc = False
            continue
        if ch == "\\":
            esc = True
            continue
        if ch == '"':
            in_str = not in_str
            continue
        if in_str:
            continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                blob = s[start:i + 1]
                try:
                    return json.loads(blob)
                except Exception:
                    return None
    return None


def extract_code_tuples(record):
    out = set()
    if not isinstance(record, dict):
        return out
    for top in ("conditions", "labs", "meds"):
        items = record.get(top) or []
        if not isinstance(items, list):
            continue
        for item in items:
            if not isinstance(item, dict):
                continue
            for system_label, key in CODE_SYSTEMS_LIST:
                v = item.get(key)
                if v is None or v == "":
                    continue
                out.add((system_label, str(v)))
    vitals = record.get("vitals") or {}
    if isinstance(vitals, dict):
        for k, v in vitals.items():
            if v is None or v == "":
                continue
            out.add(("vital", str(k)))
    return out


def f1(p, r):
    return 0.0 if (p + r) == 0 else 2 * p * r / (p + r)


# ----- run the bench --------------------------------------------------------
FastLanguageModel.for_inference(model)

with val_path.open() as fh:
    lines = [ln for ln in fh if ln.strip()]
if BENCH_LIMIT:
    lines = lines[:BENCH_LIMIT]

per_case = []
for idx, ln in enumerate(lines):
    rec = json.loads(ln)
    convs = rec["conversations"]
    sys_msg = next(c["content"] for c in convs if c["role"] == "system")
    user_msg = next(c["content"] for c in convs if c["role"] == "user")
    gold_msg = next(c["content"] for c in convs if c["role"] == "assistant")

    inputs = tokenizer.apply_chat_template(
        [{"role": "system", "content": sys_msg},
         {"role": "user", "content": user_msg}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt",
    ).to("cuda")
    t0 = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=BENCH_MAX_NEW_TOKENS,
            temperature=0.0,
            top_p=1.0,
            do_sample=False,
        )
    elapsed = time.perf_counter() - t0
    pred_text = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

    gold = try_parse_json(gold_msg) or {}
    pred = try_parse_json(pred_text)
    json_valid = pred is not None
    schema_complete = json_valid and isinstance(pred, dict) and SCHEMA_KEYS.issubset(set(pred.keys()))
    gold_codes = extract_code_tuples(gold)
    pred_codes = extract_code_tuples(pred or {})
    tp = len(gold_codes & pred_codes)
    fp = len(pred_codes - gold_codes)
    fn = len(gold_codes - pred_codes)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec_score = tp / (tp + fn) if (tp + fn) else 0.0
    case = {
        "idx": idx, "tp": tp, "fp": fp, "fn": fn,
        "precision": prec, "recall": rec_score, "f1": f1(prec, rec_score),
        "json_valid": json_valid, "schema_complete": bool(schema_complete),
        "latency_s": elapsed,
    }
    per_case.append(case)
    if idx % 10 == 0 or idx + 1 == len(lines):
        print(f"  [{idx:3d}/{len(lines)}] f1={case['f1']:.3f} json={json_valid} t={elapsed:.2f}s", flush=True)

# ----- aggregate ------------------------------------------------------------
n = len(per_case)
tp_sum = sum(c["tp"] for c in per_case)
fp_sum = sum(c["fp"] for c in per_case)
fn_sum = sum(c["fn"] for c in per_case)
micro_p = tp_sum / (tp_sum + fp_sum) if (tp_sum + fp_sum) else 0.0
micro_r = tp_sum / (tp_sum + fn_sum) if (tp_sum + fn_sum) else 0.0
micro_f1 = f1(micro_p, micro_r)
json_valid_rate = sum(1 for c in per_case if c["json_valid"]) / n
schema_rate = sum(1 for c in per_case if c["schema_complete"]) / n
latencies = sorted(c["latency_s"] for c in per_case)
def pct(p):
    k = max(0, min(len(latencies) - 1, int(round(p * (len(latencies) - 1)))))
    return latencies[k]
above_70 = sum(1 for c in per_case if c["f1"] >= 0.70)
valid_only = [c for c in per_case if c["json_valid"]]
if valid_only:
    vtp = sum(c["tp"] for c in valid_only)
    vfp = sum(c["fp"] for c in valid_only)
    vfn = sum(c["fn"] for c in valid_only)
    vp = vtp / (vtp + vfp) if (vtp + vfp) else 0.0
    vr = vtp / (vtp + vfn) if (vtp + vfn) else 0.0
    vf = f1(vp, vr)
else:
    vp = vr = vf = 0.0

bench_summary = {
    "n": n,
    "micro_precision": round(micro_p, 4),
    "micro_recall": round(micro_r, 4),
    "micro_f1": round(micro_f1, 4),
    "json_valid_rate": round(json_valid_rate, 4),
    "schema_complete_rate": round(schema_rate, 4),
    "latency_p50_s": round(pct(0.50), 3),
    "latency_p95_s": round(pct(0.95), 3),
    "cases_above_f1_0_70": above_70,
    "json_valid_only_n": len(valid_only),
    "json_valid_only_f1": round(vf, 4),
    "json_valid_only_precision": round(vp, 4),
    "json_valid_only_recall": round(vr, 4),
}
print("\n=== Inline bench summary (this kernel run) ===")
print(json.dumps(bench_summary, indent=2))


## Results — v63b LoRA (coverage-restored) vs v63 (broken k/v coverage) vs base `gemma-4-E2B-it`

The numbers below come from the offline llama.cpp bench at `apps/mobile/convert/bench_v62_singleshot.py` running on Mac M-series with both models quantised to **Q3_K_M GGUF** so the comparison is apples-to-apples on the same edge hardware.

**v63 regression (the bug this kernel fixes):** v63's Kaggle full-precision bench gave F1 = 0.9989, but after Mac Q3_K_M conversion the same adapter collapsed to **F1 = 0.5475**. Root cause: only 410 LoRA tensors were saved instead of 490 — k_proj and v_proj were silently dropped on global-attention decoder layers 15-34. Quantization rounds the partial-coverage attention into noise; full-precision attention is forgiving of it.

| Metric (val-compact, 200 cases, Q3_K_M) | Base `gemma-4-E2B-it` | + ClinIQ v63 (broken) | + ClinIQ v63b (target) |
|---|---:|---:|---:|
| Micro-F1 (code-level) | 0.337 | 0.5475 | **>= 0.85** (acceptance gate) |
| LoRA tensors saved | n/a | 410 | **>= 490** (enforced) |
| k_proj LoRA layers | n/a | 15 / 35 (local only) | **35 / 35** (enforced) |
| v_proj LoRA layers | n/a | 15 / 35 (local only) | **35 / 35** (enforced) |

The inline bench cell above re-runs the same scoring inside Kaggle on the (typically) bf16/4-bit Unsloth model. Kaggle inline numbers will look similar between v63 and v63b — full-precision attention masks the coverage gap. The real comparison happens after Mac Q3_K_M conversion, where v63 collapsed and v63b is expected to hold.

## 9. Save the LoRA adapter

The adapter goes to `cliniq_lora/` in the kernel's working directory and shows up as a downloadable artifact in the **Output** tab of this kernel. The full adapter is 124 MB at `r=16` across 7 target modules. GGUF conversion (for llama.cpp / iOS / Jetson deployment) is intentionally done **locally** with the project repo's `scripts/convert_lora_to_gguf.py` because Kaggle's T4 disk fills up during a merge+convert.


In [ ]:
# Save LoRA adapter
import os
import sys
import json
import glob

tag = "-compact" if COMPACT else ""
lora_dir = "cliniq_lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
lora_size = sum(os.path.getsize(os.path.join(lora_dir, f)) for f in os.listdir(lora_dir) if os.path.isfile(os.path.join(lora_dir, f)))
print(f"LoRA saved to {lora_dir}/ — {lora_size / 1024 / 1024:.1f} MB ({len(os.listdir(lora_dir))} files)")

# ----- v63b post-save tensor-count assertion --------------------------------
# v62 saved 490 LoRA tensors. v63 saved only 410 (k_proj + v_proj dropped on global-attention
# layers 15-34). v63b must hit >= 490, or we have shipped the same broken artifact.
try:
    from safetensors import safe_open
    tensor_keys = []
    st_files = sorted(glob.glob(os.path.join(lora_dir, "*.safetensors")))
    print(f"safetensors files: {[os.path.basename(p) for p in st_files]}")
    for p in st_files:
        with safe_open(p, framework="pt") as f:
            tensor_keys.extend(list(f.keys()))
    print(f"Total LoRA tensor count: {len(tensor_keys)} (v62 baseline = 490, v63 broken = 410)")

    # Cross-check: how many k_proj / v_proj LoRA tensors are saved (should be 35 * 2 = 70 each)?
    k_tensors = [k for k in tensor_keys if "k_proj" in k]
    v_tensors = [k for k in tensor_keys if "v_proj" in k]
    print(f"  k_proj tensors saved: {len(k_tensors)} (expected 70 = 35 layers * lora_A+lora_B)")
    print(f"  v_proj tensors saved: {len(v_tensors)} (expected 70 = 35 layers * lora_A+lora_B)")

    assert len(tensor_keys) >= 490, (
        f"v63b FATAL: saved only {len(tensor_keys)} LoRA tensors, expected >= 490. "
        f"k_proj={len(k_tensors)} v_proj={len(v_tensors)}. The v63 bug is still present."
    )
    print("v63b tensor-count check OK — adapter has full coverage.")

    # Also dump adapter_config.json so we can sanity-check layers_to_transform on the saved side.
    cfg_path = os.path.join(lora_dir, "adapter_config.json")
    if os.path.exists(cfg_path):
        with open(cfg_path) as fh:
            cfg = json.load(fh)
        print("\nadapter_config.json (relevant fields):")
        for k in ("target_modules", "layers_to_transform", "layers_pattern", "auto_mapping"):
            print(f"  {k}: {cfg.get(k)}")
except AssertionError:
    raise
except Exception as e:
    print(f"Post-save check raised non-assertion error: {e!r}", flush=True)
    raise

# GGUF export skipped — T4 doesn't have enough disk space for merge+export.
# Use convert_lora_to_gguf.py locally with the downloaded LoRA adapter:
#   python /tmp/llama-cpp-tools/convert_lora_to_gguf.py cliniq_lora/ --outfile model.gguf --base-model-id unsloth/gemma-4-E2B-it
# Then apply at runtime: llama-server -m base.gguf --lora model.gguf
print("Download cliniq_lora/ from Output tab for local GGUF conversion (v63b)")

## 10. What's novel

Three things in this submission are not the usual fine-tune-a-checkpoint story.

**1. Distillation from a 6-turn agent into single-shot.** The project's quality path is a multi-turn LangChain-style agent that does iterative SNOMED/LOINC/RxNorm lookup against an embedded RAG database (~60 verified codes) and hits **F1 = 1.000** on a 64-case combined eICR set — but takes 5-35 seconds per case. The v63 LoRA (experiment) is the **distilled single-shot** version: one forward pass, no RAG, no tool calls, and 4.05s p50 on Mac M-series (Q3_K_M) — meaningfully faster than the agent on the same hardware while preserving 0.823 micro-F1 (0.895 on the JSON-valid subset).

**2. 1.6x faster than the base model on the same edge target.** Most LoRA fine-tunes pay a latency cost. This one is *faster* than the base model at the same Q3_K_M quantisation because the LoRA learned the compact-JSON schema and produces shorter outputs (mean 165 tokens vs base's 280-ish on long-tail expansions). Latency p50 drops from 6.59s to 4.05s.

**3. Edge-deployable in ~5 GB total.** Base Gemma 4 E2B-it Q3_K_M is ~3.7 GB; the LoRA is ~124 MB; the GGUF adapter is similar. The whole inference stack — model + adapter + llama.cpp runtime + 200-case bench harness — fits in <5 GB and runs on a 2020 M1 MacBook Air, a Jetson Orin Nano (8 GB), or an iPhone 15 Pro through `LiteRT-LM`. **Precision is 0.979.** That's strong enough to anchor a clinic-side pre-screening UI where every flagged code is shown to a human for confirmation.

The combination — distilled, faster than the base model, fits on a phone, 0.98 precision — is what we mean by "good" in the Gemma 4 *Good* Hackathon framing.


## 11. Limitations, future v63 path, and citations

### Limitations

- **Synthetic training data.** The 800 train + 200 val examples are clinician-formatted pseudo-eICRs (`Patient: ...\nDx: ...\nLab: ...\n`), not full HL7 CDA XML. Real eICRs include a CDA wrapper, structured code tables, and free-text narrative. The repo's free-text-narrative path (regex + RAG + 6-turn agent) handles those at F1 = 1.000; this LoRA is the **fast single-shot** path for known-format inputs.
- **86% JSON validity.** The 14% gap is entirely length-limit truncation at `max_new_tokens = 2048` — the extraction is good, but the closing brace gets cut off. Mitigations: client-side retry on `JSONDecodeError`, longer `max_new_tokens` budget, or a v63 retrain with longer `max_seq_length` and ~50 long-expansion examples.
- **GBNF grammar regressed F1.** Grammar-constrained decoding pushed paths into longer expansions that hit the length limit even more often. We ship without grammar.
- **English only**, US clinical conventions (SNOMED CT, ICD-10-CM, LOINC, RxNorm). No localisation for ICD-11 or non-US ontologies.
- **No PHI in training data** — synthetic patients only. The model has not seen real protected information; downstream deployment should still treat outputs as PHI-equivalent because the **input** is.
- **Compact JSON != FHIR Bundle.** The output needs one transform layer (`apps/mobile/convert/fhir_bundle.py` in the repo) to become a R4 Bundle.

### Future v63 path

The shortest path to a meaningful step-up:
1. Bump `max_seq_length` to 1024 to eliminate truncation.
2. Add ~50 long-expansion training examples (multi-condition, multi-lab, multi-med cases) — these are the cases that currently truncate.
3. Retain the same 5-epoch / lr=1e-4 / `packing=True` recipe.

Expected outcome from the JSON-valid-only number: **F1 ~0.90**, JSON validity ~95%, no latency regression. This is tracked in the project repo but **not part of this submission** to keep the submitted artefact reproducible from a single Kaggle kernel.

### Artifacts

- **Public Kaggle notebook**: this kernel.
- **HF Hub model card**: `kcirtapfromspace/cliniq-gemma4-e2b-unsloth-v62` (see repo `tools/autoresearch/v62-submission/MODEL_CARD.md`).
- **Source repo**: `github.com/patrickdeutsch/gemma4`.
- **iOS app**: `apps/mobile/ios-app/ClinIQ/`.
- **Hosted demo**: `huggingface.co/spaces/kcirtapfromspace/cliniq-eicr-fhir`.
- **Bench harness**: `apps/mobile/convert/bench_v62_singleshot.py`.
- **Bench output (this submission)**: `apps/mobile/convert/build/v62_val_compact_bench.json`.
- **Submission narrative**: `tools/autoresearch/hackathon-submission-2026-04-27.md`.

### Citation

```bibtex
@misc{cliniq-gemma4-2026,
  author = {Patrick Deutsch},
  title  = {ClinIQ: Gemma 4 E2B fine-tuned with Unsloth for offline eICR extraction},
  year   = {2026},
  note   = {Gemma 4 Good Hackathon — Unsloth Track submission},
}
```

### Acknowledgements

- **Unsloth** (Daniel Han, Michael Han) for the 4-bit fine-tuning stack and the verified `gemma-4` chat template.
- **Google DeepMind** for Gemma 4 E2B-it.
- **CDC** for the RCTC reportable-condition list and the EZeCR architecture that motivated this work.
- **HL7 / FHIR community** for the R4 schema this output is designed to bundle into.

License: Apache 2.0 (LoRA adapter); base model under the Gemma Terms of Use.
